# 07 — Real-data baseline: currently-available models on real GReX

Runs the **currently-available models** — elastic net + the three pre-existing Bayesian
priors (`regularized_horseshoe`, `horseshoe`, `bayesian_lasso`) — against the real cohort's
GReX + phenotype files, the same models exercised on simulated data in notebooks 02-04. This
is a sanity check that real data flows correctly through the existing pipeline *before* any
structured-prior model (`graph_horseshoe`, `group_horseshoe`, `spike_slab`, `graph_spike_slab`)
is tried on it (that comes in a later notebook, once sub-goal 3's biological-pathway/graph
source is decided — see `docs/ROADMAP.md`).

> **Data-safety reminder:** this notebook is meant to be filled in and *run in the secured
> environment* (the `cloud-dev-workflow` pattern) — real file paths/column names below, real
> data never leaves that environment. Clear all cell outputs before this notebook is ever
> committed/pushed anywhere; only code should leave the secured system, errors reported back
> as text.


## Config — **EDIT THIS CELL** before running

In [ ]:
# ==== EDIT ME: point these at the real files, in the secured environment ====
GREX_PATH = "/path/to/real_grex.tsv"                  # EDIT ME
PHENO_PATH = "/path/to/real_phenotype.tsv"            # EDIT ME
SAMPLE_COL = "GRID"                                   # EDIT ME if this cohort isn't VUMC (e.g. AoU's own ID column)
TRAIT = "EDIT_ME_trait_column_name"                   # EDIT ME — the trait column in the phenotype file
COVAR_COLS = ["EDIT_ME_covar_1", "EDIT_ME_covar_2"]   # EDIT ME — or [] / None for no covariates
FAMILY = "gaussian"                                   # EDIT ME: "gaussian" (continuous trait) or "binomial" (case/control coded 1/0)
GREX_UNCERTAINTY_PATH = None                          # optional; leave None until this is available (see ROADMAP.md sub-goal 1)
OUTER_K = 5                                           # nested-CV outer folds; lower this first if real n_genes makes MCMC slow
NUM_WARMUP = NUM_SAMPLES = 300                        # MCMC length for the Bayesian arms; lower first if runtime is a problem

if any("EDIT_ME" in str(v) or "path/to" in str(v) for v in (GREX_PATH, PHENO_PATH, TRAIT, COVAR_COLS)):
    raise ValueError(
        "Fill in GREX_PATH, PHENO_PATH, SAMPLE_COL, TRAIT, COVAR_COLS, FAMILY above "
        "before running the rest of this notebook."
    )


## Load the real dataset

Uses `io.load_dataset` (see `docs/ROADMAP.md`'s "Real-data I/O Q&A" for what's already been
confirmed about this cohort's file shapes). Warns if the GReX and phenotype sample sets don't
match exactly; always keeps their intersection.

In [ ]:
import ptgs_bc as ptgs
from ptgs_bc import (load_dataset, ElasticNetBuilder, BayesBuilder,
                     run_benchmark, summary_table, per_fold_table, plot_performance)
%matplotlib inline

ds = load_dataset(
    GREX_PATH, PHENO_PATH,
    trait=TRAIT, covar_cols=(COVAR_COLS or None),
    family=FAMILY, sample_col=SAMPLE_COL,
    grex_uncertainty_path=GREX_UNCERTAINTY_PATH,
)
print(ds.n_samples, "samples x", ds.n_genes, "genes | family:", ds.family)
print("covariates:", list(ds.covars.columns) if ds.covars is not None else None)
print("grex_uncertainty loaded:", "grex_uncertainty" in ds.meta)


## The currently-available models (baseline — no structured priors yet)

Same four arms as notebooks 02-04: elastic net, and the three pre-existing (iid-shrinkage)
Bayesian priors. `BayesBuilder` already adjusts for `ds.covars` internally (see
`builders/bayes.py`) — no extra wiring needed here.

In [ ]:
builders = [
    ElasticNetBuilder(),
    BayesBuilder(prior="regularized_horseshoe", num_warmup=NUM_WARMUP, num_samples=NUM_SAMPLES),
    BayesBuilder(prior="horseshoe", num_warmup=NUM_WARMUP, num_samples=NUM_SAMPLES),
    BayesBuilder(prior="bayesian_lasso", num_warmup=NUM_WARMUP, num_samples=NUM_SAMPLES),
]


## Run through the same nested CV

In [ ]:
res = run_benchmark(builders, ds, outer_k=OUTER_K, seed=0)
summary_table(res)


## Per-fold table + comparison figure

In [ ]:
pf = per_fold_table(res)
display(pf.pivot(index="fold", columns="builder", values="value"))
fig = plot_performance(res)
fig


## Next step

If these baseline arms ran cleanly on the real data, the next notebook(s) bring in the
structured priors (`graph_horseshoe`, `group_horseshoe`, `spike_slab`, `graph_spike_slab`)
using real biological pathway/graph input in place of the oracle/synthetic structure from
Phase 1.5/1.6 — pending sub-goal 3's discussion of which database(s) to use.

If something broke here instead, report the error back (as text — cloud-dev-workflow) so it
can be fixed without any real data leaving the secured environment.